# Fraud 3 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Travel booking payments

**Problem:** Identify suspicious travel payments before ticket fulfilment.

**Proposed success criterion:** Compare with an all-legitimate baseline; justify a threshold under a 6% review-capacity constraint; discuss customer friction.

**Dataset:** `fraud_3.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


## Phase 1 — Business Understanding and Phase 2 — Data Understanding (review)
Explain the decision, stakeholder, prediction moment, observation unit, key and target. **Review or revise the supplied success criterion if it does not adequately represent the business decision.** State two evidenced findings and two limitations.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('fraud_3.csv')
if not DATA.exists(): DATA = Path('fraud_3.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(2601, 12)


,transaction_id,amount_eur,customer_avg_amount_30d,account_age_days,previous_transactions_24h,channel,country,hour,fraud_flag,post_investigation_loss_eur,days_to_departure,refund_after_trip_eur
0,T6000000,NaN,43.58,1357,4,ONLINE,FR,16,0,0.0,73,0.0
1,T6000001,17.82,43.80,1983,15,POS,FR,9,0,0.0,46,0.0
2,T6000002,143.45,31.24,612,12,Online,DE,9,0,0.0,5,0.0
3,T6000003,47.77,17.09,194,15,online,FR,9,0,0.0,167,0.0
4,T6000004,20.90,14.42,1622,5,Mobile,NL,13,0,0.0,199,0.0


amount_eur                     11
transaction_id                  0
customer_avg_amount_30d         0
account_age_days                0
previous_transactions_24h       0
channel                         0
country                         0
hour                            0
fraud_flag                      0
post_investigation_loss_eur     0
days_to_departure               0
refund_after_trip_eur           0
dtype: int64

Exact duplicates: 1


**phase 1**

Stakeholders: Manager, the customer, customer service, finance team.
Prediction moment: when you authorize for the payment. 



**One row represents:** one transaction 
**Dimensions:**   2600 entries/12 variables
**Identifier/business key:** transaction_id  
**Main outcome:** fraud_flag 
**Most relevant variables and why:** It is country and transaction in last 24H because it is not possible to make a transaction in the Netherlands and Spain within 2 hours of eachother. 

**Revision of the succes criterion**
Fraud is only 2.23% of the total transactions and the cap that the company can handle is 6% meaning we can increase the treshold on the fraud flag to get flag more transactions without going over the limit. 

**2 findings and 2 limitations**
2 findings:
1. fraud is only a small percentage of the total transactions.
2. post_investigation_loss_euro can not be used in the model because it says if a transaction is fraud or not but only AFTER it happened.

2 limitations:
1. The data ons fraud is small and there are some issues in the dataset
2. there is no information on what a flase positive costs the company.

### Data-quality findings
| Observation/issue | Variable(s) | Evidence | Why it matters |
|Missing|amount_eur|11|If there is fraud we do not know how much to reimburse the customer.|
|Duplicate|transaction_id |2| It changes the statistics of the dataset and we do now know if it is an error or not.|
|Capital letter |channel |online/Online/ONLINE | There are different ways that it says online in the dataset, this could mean that the code doesnt read certain lines and that changes the output.|
| | | | |

## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [8]:
work = df.copy()

In [9]:
# [1] Remove the exact duplicate row
before_rows = work.shape[0]
work = work.drop_duplicates()
after_rows = work.shape[0]
print(f"[1] Duplicates removed: {before_rows - after_rows}  (before={before_rows}, after={after_rows})")

# [2] Standardise inconsistent 'channel' categories (same channel, different capitalisation)
print("\n[2] channel BEFORE:")
print(work['channel'].value_counts())
work['channel'] = work['channel'].str.strip().str.capitalize()
print("\n[2] channel AFTER:")
print(work['channel'].value_counts())

# [3] Handle missing amount_eur values: keep a missing-flag, then impute with the median
missing_before = work['amount_eur'].isna().sum()
work['amount_eur_missing'] = work['amount_eur'].isna().astype(int)
median_amount = work['amount_eur'].median()
work['amount_eur'] = work['amount_eur'].fillna(median_amount)
print(f"\n[3] amount_eur missing BEFORE: {missing_before}  AFTER: {work['amount_eur'].isna().sum()}  (filled with median={median_amount:.2f})")

# [4] New feature (feature engineering): how far the amount is from the customer's usual spend
work['amount_to_avg_ratio'] = work['amount_eur'] / work['customer_avg_amount_30d'].replace(0, np.nan)
work['amount_to_avg_ratio'] = work['amount_to_avg_ratio'].fillna(work['amount_to_avg_ratio'].median())
print("\n[4] Mean ratio by fraud_flag:")
print(work.groupby('fraud_flag')['amount_to_avg_ratio'].mean().round(2))

# [5] Exclude identifier and leakage columns from the model (not deleted, just not used as predictors)
excluded_cols = ['transaction_id', 'post_investigation_loss_eur', 'refund_after_trip_eur']
print("\n[5] Columns excluded from the model (identifier / leakage):", excluded_cols)

work.head()

[1] Duplicates removed: 1  (before=2601, after=2600)

[2] channel BEFORE:
channel
Mobile    661
POS       654
online    644
Online    634
ONLINE      7
Name: count, dtype: int64

[2] channel AFTER:
channel
Online    1285
Mobile     661
Pos        654
Name: count, dtype: int64

[3] amount_eur missing BEFORE: 11  AFTER: 0  (filled with median=47.52)

[4] Mean ratio by fraud_flag:
fraud_flag
0    2.41
1    3.93
Name: amount_to_avg_ratio, dtype: float64

[5] Columns excluded from the model (identifier / leakage): ['transaction_id', 'post_investigation_loss_eur', 'refund_after_trip_eur']


,transaction_id,amount_eur,customer_avg_amount_30d,account_age_days,previous_transactions_24h,channel,country,hour,fraud_flag,post_investigation_loss_eur,days_to_departure,refund_after_trip_eur,amount_eur_missing,amount_to_avg_ratio
0,T6000000,47.52,43.58,1357,4,Online,FR,16,0,0.0,73,0.0,1,1.090408
1,T6000001,17.82,43.80,1983,15,Pos,FR,9,0,0.0,46,0.0,0,0.406849
2,T6000002,143.45,31.24,612,12,Online,DE,9,0,0.0,5,0.0,0,4.591869
3,T6000003,47.77,17.09,194,15,Online,FR,9,0,0.0,167,0.0,0,2.795202
4,T6000004,20.90,14.42,1622,5,Mobile,NL,13,0,0.0,199,0.0,0,1.449376


### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
| 1 exact duplicate row (`transaction_id`) | Removed with `drop_duplicates()` | Rows: 2,601 → 2,600 | If it were a real repeated purchase we'd lose data, but identical values make an error more likely | Keep the deduplicated set |
| `channel` had 5 inconsistent labels (case only) | Standardised with `.str.capitalize()` | Categories: 5 → 3 (`Mobile`, `Online`, `Pos`) | Assumes casing was the only difference (confirmed) | One-hot encode the clean 3 categories |
| `amount_eur` had 11 missing values | Added a missing-flag + imputed with the median (€47.52) | Missing: 11 → 0 | Imputation hides the real value and why it was missing | Keep the flag as its own feature |
| `post_investigation_loss_eur` and `refund_after_trip_eur` reveal the outcome / are post-decision | Excluded from the model entirely, along with `transaction_id` | Feature list built without these 3 columns | None — this prevents leakage | Model only with pre-decision information |

## Phase 4 — Modelling
Define the prediction moment and allowed predictors. Use a defensible train/test split (chronological for forecasting). Fit learned preprocessing on TRAINING data only. Compare a simple baseline with a candidate model. Explain your feature engineering and excluded fields.

In [10]:
# Build and test your baseline and candidate model here.
# Suggested tools: sklearn Pipeline, ColumnTransformer, SimpleImputer,
# OneHotEncoder, LogisticRegression, RandomForestRegressor or Ridge.
# For forecasting: use time-ordered validation; no future sales or actual future footfall.
# Phase 4 — Modelling
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

target = 'fraud_flag'
numeric_features = ['amount_eur','customer_avg_amount_30d','account_age_days',
                     'previous_transactions_24h','hour','days_to_departure','amount_to_avg_ratio']
categorical_features = ['channel','country']
# transaction_id, post_investigation_loss_eur, refund_after_trip_eur are excluded (leakage, Phase 3)

X = work[numeric_features + categorical_features]
y = work[target]

# Stratified split: fraud is only ~2.2% of the data, so stratify keeps that ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} rows, fraud rate {y_train.mean():.4f}")
print(f"Test:  {X_test.shape[0]} rows, fraud rate {y_test.mean():.4f}  ({y_test.sum()} fraud cases)")

# Preprocessing is fit ONLY on training data (inside the Pipeline), to avoid leakage
preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# --- Baseline: always predict "legitimate" ---
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print("\n=== Baseline: always predict legitimate ===")
print(confusion_matrix(y_test, baseline_pred))
print(classification_report(y_test, baseline_pred, digits=3, zero_division=0))

# --- Candidate model: Logistic Regression, class-weighted for the imbalance ---
candidate = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
candidate.fit(X_train, y_train)
candidate_pred = candidate.predict(X_test)
candidate_proba = candidate.predict_proba(X_test)[:, 1]

print("\n=== Candidate: Logistic Regression (class-weighted), default 0.5 threshold ===")
print(confusion_matrix(y_test, candidate_pred))
print(classification_report(y_test, candidate_pred, digits=3, zero_division=0))
print("ROC AUC:", round(roc_auc_score(y_test, candidate_proba), 3))

# --- Review-capacity check: review only the top 6% riskiest transactions ---
review_capacity = 0.06
n_review = max(1, int(np.ceil(len(X_test) * review_capacity)))
ranked = pd.DataFrame({'y_true': y_test.values, 'proba': candidate_proba}).sort_values('proba', ascending=False)
reviewed = ranked.head(n_review)
fraud_caught = int(reviewed['y_true'].sum())
total_fraud_in_test = int(y_test.sum())
print(f"\n=== Review-capacity check: reviewing the top {review_capacity:.0%} ({n_review} of {len(X_test)} test transactions) ===")
print(f"Fraud cases caught: {fraud_caught} of {total_fraud_in_test} total fraud in the test set "
      f"({fraud_caught/total_fraud_in_test:.1%} recall at {review_capacity:.0%} capacity)")

Train: 1820 rows, fraud rate 0.0225
Test:  780 rows, fraud rate 0.0218  (17 fraud cases)

=== Baseline: always predict legitimate ===
[[763   0]
 [ 17   0]]
              precision    recall  f1-score   support

           0      0.978     1.000     0.989       763
           1      0.000     0.000     0.000        17

    accuracy                          0.978       780
   macro avg      0.489     0.500     0.494       780
weighted avg      0.957     0.978     0.967       780


=== Candidate: Logistic Regression (class-weighted), default 0.5 threshold ===
[[481 282]
 [  9   8]]
              precision    recall  f1-score   support

           0      0.982     0.630     0.768       763
           1      0.028     0.471     0.052        17

    accuracy                          0.627       780
   macro avg      0.505     0.550     0.410       780
weighted avg      0.961     0.627     0.752       780

ROC AUC: 0.52

=== Review-capacity check: reviewing the top 6% (47 of 780 test transac

### Phase 4 — Modelling: written answer

**Prediction moment and allowed predictors:** The prediction happens at payment authorisation, before the ticket is issued. Allowed predictors are listed in `numeric_features` and `categorical_features` in the code above; `transaction_id`, `post_investigation_loss_eur` and `refund_after_trip_eur` are excluded because they are only known after the outcome.

**Train/test split:** We used a stratified 70/30 split (not chronological, since this is transaction classification, not forecasting). Stratifying was necessary because fraud is only 2.2% of the data — without it, the test set could randomly end up with very few or no fraud cases. The output confirms both sets kept a similar fraud rate (train 2.25%, test 2.18%).

**Preprocessing fit on training data only:** The `ColumnTransformer`  is placed inside the `Pipeline` and only called via `candidate.fit(X_train, y_train)`, so it learns its parameters exclusively from the training set, never from the test set.

**Baseline vs. candidate:** The baseline reaches 97.8% accuracy but 0% recall, it catches no fraud at all. The candidate Logistic Regression catches more fraud (47% recall) but with a very high false-positive cost (282 of 763 legitimate transactions flagged), and only a 0.52 ROC AUC, barely better than random.

**Feature engineering and excluded fields:** We engineered `amount_to_avg_ratio` (amount divided by the customer's 30-day average spend) in Phase 3, which showed a clear difference between fraud and legitimate transactions (mean 3.93 vs 2.41). `transaction_id`, `post_investigation_loss_eur` and `refund_after_trip_eur` were excluded as they are not available at the prediction moment.

## Phase 5 — Evaluation
Report model vs baseline on held-out observations. Interpret relevant classification errors or forecasting MAE by business segment. Compare with your Phase 1 success criterion and discuss whether that criterion is realistic. If a required cost/impact is unavailable, state the assumption or request more data.

## Phase 6 — Deployment proposal (design only)
Specify who uses the prediction, when it runs, what action it supports, a human override, monitoring metric, feedback/refresh trigger and a condition for pausing use. A diagram or 6–8 sentences is sufficient. No production system required.

## Final group decision + individual accountability
**Group:** Pilot / Improve / Revisit earlier CRISP-DM phase / Do not deploy yet. Cite evidence.

**Individual:** Every member adds a named Markdown paragraph describing one preparation choice, one model finding and one business implication.